# 10 – Parcel Geometry Cleaning (Maricopa County)

This notebook focuses on **parcel geometries** for Maricopa County.

**Goals:**
- Load the raw parcel shapefile(s).
- Inspect and confirm the CRS.
- Reproject to the project CRS (if needed).
- Do basic geometry checks (null geometries, invalid shapes).
- Save a cleaned, standardized parcel layer for later use.

In [1]:
from pathlib import Path
import geopandas as gpd

# Assume notebooks live either in project root or in a 'notebooks' folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PARCELS_DIR = DATA_DIR / "parcels"

# We will read a raw parcel shapefile and later write out a cleaned version
PARCELS_DIR

WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels')

In [14]:
raw_parcels_path = PARCELS_DIR / "Parcels_20241108.shp"

clean_parcels_path = PARCELS_DIR / "processed/maricopa_parcels_cleaned.gpkg"

raw_parcels_path, clean_parcels_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels/Parcels_20241108.shp'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels/processed/maricopa_parcels_cleaned.gpkg'))

In [3]:
# Load the raw parcel data
parcels = gpd.read_file(raw_parcels_path)

print("Number of features:", len(parcels))
print("CRS:", parcels.crs)

parcels.head()

Number of features: 1736192
CRS: EPSG:2868


,APN,Floor,LandSize,StartDate,Shp_Area,Shp_Length,geometry
0,10101001C,1,7565.0,2008-09-22,7505.147490,350.125679,"POLYGON Z ((581182.478 886654.38 0, 581182.177..."
1,10101010,1,195864.0,2008-09-22,195989.095940,1838.766520,"POLYGON Z ((582322.095 889960.815 0, 582338.88..."
2,10101011,1,95491.0,2008-09-22,95501.799733,1248.957937,"POLYGON Z ((581500.432 889688.576 0, 581324.69..."
3,10101012,1,66960.0,2008-09-22,66976.092596,1014.388578,"POLYGON Z ((581428.706 889618.792 0, 581414.67..."
4,10101014,1,126838.0,2008-09-22,126838.044773,1424.568514,"POLYGON Z ((581359.726 889391.699 0, 581362.74..."


In [4]:
PROJECT_CRS = "EPSG:2868"
PROJECT_CRS

'EPSG:2868'

In [5]:
if PROJECT_CRS is not None and parcels.crs != PROJECT_CRS:
    parcels = parcels.to_crs(PROJECT_CRS)
    print("Reprojected parcels to PROJECT_CRS:", PROJECT_CRS)
else:
    print("No reprojection performed. Check PROJECT_CRS and parcels.crs.")

No reprojection performed. Check PROJECT_CRS and parcels.crs.


In [ ]:
# Check for null geometries
null_geom_count = parcels["geometry"].isna().sum()
print("Null geometries:", null_geom_count)

# Optionally, drop rows with null geometry
# parcels = parcels[~parcels["geometry"].isna()].copy()

# Check for invalid geometries
invalid_mask = ~parcels.is_valid
invalid_count = invalid_mask.sum()
print("Invalid geometries:", invalid_count)

# Try to fix invalid geometries using buffer(0) (thanks to Copilot for the suggestion)
parcels.loc[invalid_mask, "geometry"] = parcels.loc[invalid_mask, "geometry"].buffer(0)
print("Invalid geometries after buffer(0):", (~parcels.is_valid).sum())

Null geometries: 0
Invalid geometries: 73
Invalid geometries after buffer(0): 0


In [8]:
parcels.columns

Index(['APN', 'Floor', 'LandSize', 'StartDate', 'Shp_Area', 'Shp_Length',
       'geometry'],
      dtype='object')

In [10]:
parcels[["APN","LandSize", "Floor"]].head()

,APN,LandSize,Floor
0,10101001C,7565.0,1
1,10101010,195864.0,1
2,10101011,95491.0,1
3,10101012,66960.0,1
4,10101014,126838.0,1


In [15]:
# Save cleaned parcels to a GeoPackage
parcels.to_file(clean_parcels_path, driver="GPKG")
print("Saved cleaned parcels to:", clean_parcels_path)

Saved cleaned parcels to: c:\Users\arjav\DevSource\toc-performance-dashboard\data\parcels\processed\maricopa_parcels_cleaned.gpkg
